## CAI 501/AIE 501: AI Systems

##### *Assignment 0: Understanding DNN through MLP vs. CNN on MNIST*

In [ ]:
import copy
import os
import platform
import time
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from scipy.stats import binomtest
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST

##### Experiment Map
MLP: 784 → 256 → 128 → 10, fully connected, ReLU

CNN: two blocks of 3 × 3 convolution + ReLU + 2 × 2 max-pooling (16 and 32 channels), then 1568 → 64 → 10

Both: same data split, same batch order, Adam (learning rate 0.001), batch 128, 15 epochs, cross-entropy, best validation epoch kept

## Step Zero: Setup and Data Reading

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

EPOCHS = 15
BATCH_SIZE = 128
LR = 1e-3
VAL_SIZE = 5000
DATA_DIR = "data"
RESULT_DIR = "Results"
os.makedirs(RESULT_DIR, exist_ok=True)
pd.set_option("display.max_colwidth", None)

print("Device:", device)

#### MNIST: the 60,000 training images are split once into 55,000 training and 5,000 validation images; the 10,000 test images stay untouched

In [ ]:
train_full = MNIST(DATA_DIR, train=True, download=True)
test_set = MNIST(DATA_DIR, train=False, download=True)

perm = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(SEED))
val_idx, train_idx = perm[:VAL_SIZE], perm[VAL_SIZE:]

X_all = train_full.data.float().div(255).unsqueeze(1)  # (60000, 1, 28, 28), pixels in [0, 1]
X_train, y_train = X_all[train_idx], train_full.targets[train_idx]
X_val, y_val = X_all[val_idx], train_full.targets[val_idx]
X_test, y_test = test_set.data.float().div(255).unsqueeze(1), test_set.targets
y_true = y_test.numpy()

for name, y in (("Training", y_train), ("Validation", y_val), ("Testing", y_test)):
    print(f"{name:<11} images: {len(y):>6}")

#### Normalization statistics from the training split only (no information from validation or test)

In [ ]:
MEAN, STD = X_train.mean().item(), X_train.std().item()
print(f"Mean: {MEAN:.4f}   Std: {STD:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for digit, ax in enumerate(axes):
    i = (y_test == digit).nonzero()[0].item()
    ax.imshow(X_test[i, 0], cmap="gray")
    ax.set_title(f"{digit} (test #{i})", fontsize=8)
    ax.axis("off")
plt.show()

## 2. Architecture Design and Understanding

### Design and implement an MLP and a CNN independently

In [ ]:
class Normalize(nn.Module):
    def forward(self, x):
        return (x - MEAN) / STD  # training-split statistics, inside the model so every test sees the full pipeline


def build_mlp():
    return nn.Sequential(OrderedDict([
        ("norm", Normalize()),
        ("flatten", nn.Flatten()),
        ("fc1", nn.Linear(784, 256)),
        ("relu1", nn.ReLU()),
        ("fc2", nn.Linear(256, 128)),
        ("relu2", nn.ReLU()),
        ("fc3", nn.Linear(128, 10)),  # logits; softmax at prediction time
    ])).to(device)


def build_cnn():
    return nn.Sequential(OrderedDict([
        ("norm", Normalize()),
        ("conv1", nn.Conv2d(1, 16, kernel_size=3, padding=1)),
        ("relu1", nn.ReLU()),
        ("pool1", nn.MaxPool2d(2)),
        ("conv2", nn.Conv2d(16, 32, kernel_size=3, padding=1)),
        ("relu2", nn.ReLU()),
        ("pool2", nn.MaxPool2d(2)),
        ("flatten", nn.Flatten()),
        ("fc1", nn.Linear(32 * 7 * 7, 64)),
        ("relu3", nn.ReLU()),
        ("fc2", nn.Linear(64, 10)),  # logits
    ])).to(device)


torch.manual_seed(SEED)
mlp = build_mlp()
torch.manual_seed(SEED)
cnn = build_cnn()
MODELS = {"MLP": mlp, "CNN": cnn}

### 1. A complete architecture diagram illustrating the data flow from input to output

In [ ]:
def layer_outputs(model, x):
    outs = [("input", x)]
    for name, layer in model.named_children():
        x = layer(x)
        outs.append((name, x))
    outs.append(("softmax", x.softmax(1)))
    return outs


COLORS = {"Linear": "#9ecae1", "Conv2d": "#a1d99b", "ReLU": "#fdd0a2", "MaxPool2d": "#dadaeb", "softmax": "#fcbba1"}


def draw_architecture(model, title):
    children = dict(model.named_children())
    with torch.no_grad():
        outs = layer_outputs(model.eval(), X_test[:1].to(device))
    fig, ax = plt.subplots(figsize=(1.25 * len(outs), 2.2))
    for i, (name, out) in enumerate(outs):
        kind = type(children[name]).__name__ if name in children else name
        ax.add_patch(plt.Rectangle((i * 1.25, 0), 1, 1, facecolor=COLORS.get(kind, "#eeeeee"), edgecolor="black"))
        ax.text(i * 1.25 + 0.5, 0.62, name, ha="center", fontsize=9, weight="bold")
        ax.text(i * 1.25 + 0.5, 0.3, "×".join(map(str, out.shape[1:])), ha="center", fontsize=8)
        if i:
            ax.annotate("", xy=(i * 1.25, 0.5), xytext=(i * 1.25 - 0.25, 0.5), arrowprops=dict(arrowstyle="->"))
    ax.set_xlim(-0.1, 1.25 * len(outs))
    ax.set_ylim(-0.1, 1.1)
    ax.axis("off")
    ax.set_title(title)
    plt.show()


draw_architecture(mlp, "MLP data flow (output shape of each layer, batch dimension omitted)")
draw_architecture(cnn, "CNN data flow (output shape of each layer, batch dimension omitted)")

### 2. The purpose and operation of every layer; 3. its input and output dimensions; 4. its trainable parameters, and the total

In [ ]:
PURPOSE = {
    "Normalize": "Standardize pixels: (x - mean) / std with training-split statistics",
    "Flatten": "Reshape into one vector; the 2D layout is discarded",
    "Linear": "Weighted sum of every input plus a bias: y = Wx + b",
    "Conv2d": "Slide shared 3 × 3 kernels over the input: local pattern detection at every position",
    "ReLU": "max(0, z): non-linearity that zeroes negative responses",
    "MaxPool2d": "Keep the maximum of each 2 × 2 window: halves height and width",
    "softmax": "Turn the 10 logits into class probabilities that sum to 1",
}


def layer_table(model):
    children = dict(model.named_children())
    with torch.no_grad():
        outs = layer_outputs(model.eval(), X_test[:1].to(device))
    rows = []
    for (_, x_in), (name, x_out) in zip(outs[:-1], outs[1:]):
        kind = type(children[name]).__name__ if name in children else name
        rows.append({"Layer": name, "Type": kind, "Input shape": tuple(x_in.shape[1:]), "Output shape": tuple(x_out.shape[1:]),
                     "Trainable parameters": sum(p.numel() for p in children[name].parameters()) if name in children else 0,
                     "Purpose": PURPOSE[kind]})
    return pd.DataFrame(rows).set_index("Layer")


for name, model in MODELS.items():
    table = layer_table(model)
    table.to_csv(f"{RESULT_DIR}/{name.lower()}_layers.csv")
    print(f"{name} total trainable parameters: {table['Trainable parameters'].sum():,}")
    display(table)

### 5. The mathematical operations performed by the main architectural components

#### Dense (fully connected) layer: every output depends on every input
$$\mathbf{h} = W\mathbf{x} + \mathbf{b}, \qquad W \in \mathbb{R}^{n_{out} \times n_{in}}, \qquad P = n_{in}\,n_{out} + n_{out}$$

#### Convolution (3 × 3, padding 1): the same kernel $W_c$ is applied at every position $(i, j)$
$$y_{c,i,j} = b_c + \sum_{k=1}^{C_{in}} \sum_{u=-1}^{1} \sum_{v=-1}^{1} W_{c,k,u,v}\; x_{k,\,i+u,\,j+v}, \qquad P = C_{out}\,(C_{in} \cdot 3 \cdot 3 + 1)$$

#### ReLU activation
$$\mathrm{ReLU}(z) = \max(0, z)$$

#### 2 × 2 max-pooling
$$p_{c,i,j} = \max_{u, v \in \{0, 1\}} a_{c,\,2i+u,\,2j+v}$$

#### Softmax output and cross-entropy loss
$$\hat{p}_k = \frac{e^{z_k}}{\sum_{j=0}^{9} e^{z_j}}, \qquad \mathcal{L} = -\log \hat{p}_{y}$$

### 6. The rationale behind your architectural choices
> MLP: two hidden layers (256 and 128 units) give non-linear capacity at a small size; the width halves at each step to compress the 784 pixels toward 10 classes.

> CNN: 3 × 3 kernels with padding keep the 28 × 28 size; two convolution and pooling blocks grow the receptive field to 10 × 10 pixels (enough to cover stroke parts) while channels grow from 16 to 32 as resolution falls.

> Both: ReLU and no dropout, weight decay or batch normalization, so the comparison isolates the architecture; the CNN is deliberately given fewer parameters than the MLP to test whether more parameters means better results (Section 7).

### Fundamental differences: spatial relationships, feature extraction and parameter sharing

In [ ]:
conv1_params = sum(p.numel() for p in cnn.conv1.parameters())
dense_equivalent = (28 * 28) * (16 * 28 * 28) + 16 * 28 * 28  # a dense layer with the same 16 × 28 × 28 output as conv1
print(f"conv1 parameters:                        {conv1_params:>12,}")
print(f"Dense layer with the same output size:   {dense_equivalent:>12,}")
print(f"Positions that reuse each conv1 kernel:  {28 * 28:>12,}")

x = X_test[:1].to(device)
shifted = torch.roll(x, shifts=3, dims=3)  # move the digit 3 pixels right
with torch.no_grad():
    a, b = cnn.conv1(cnn.norm(x)), cnn.conv1(cnn.norm(shifted))
inner = slice(4, -4)  # columns away from the wrap-around and the zero padding
print("conv1 is translation equivariant (shifted input gives the shifted feature map):",
      torch.allclose(torch.roll(a, 3, dims=3)[..., inner], b[..., inner], atol=1e-4))

> Spatial relationships: the MLP flattens the image, so neighbouring pixels are just unrelated inputs (it would learn a fixed shuffle of the pixels equally well); the CNN keeps the 2D grid and combines only neighbouring pixels.

> Feature extraction: every MLP hidden unit looks at all 784 pixels at once, a whole-image template; CNN kernels detect small local patterns that later layers combine into larger parts.

> Parameter sharing: each CNN kernel is reused at every position (160 parameters for conv1 against about 9.8 million for a dense layer with the same output), which also makes convolution translation equivariant, as checked above.

## 3. Training Process and Learning Behavior

### Train both models: same split, batch order, optimizer and epochs; the best validation epoch is kept

In [ ]:
def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


@torch.no_grad()
def predict(model, X, batch_size=1000):
    model.eval()
    return torch.cat([model(X[i:i + batch_size].to(device)).softmax(1).cpu() for i in range(0, len(X), batch_size)])


def evaluate(model, X, y):
    probs = predict(model, X)
    return F.nll_loss(probs.clamp_min(1e-12).log(), y).item(), (probs.argmax(1) == y).float().mean().item()


def train(model):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,
                              generator=torch.Generator().manual_seed(SEED))  # same batch order for both models
    history, best_acc, best_epoch, best_state = [], 0.0, 0, None
    sync()
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, correct, seen = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            seen += len(y)
        val_loss, val_acc = evaluate(model, X_val, y_val)
        history.append({"epoch": epoch, "train_loss": loss_sum / seen, "train_acc": correct / seen, "val_loss": val_loss, "val_acc": val_acc})
        if val_acc > best_acc:
            best_acc, best_epoch, best_state = val_acc, epoch, copy.deepcopy(model.state_dict())
        print(f"Epoch {epoch:>2}   train loss {loss_sum / seen:.4f}  acc {correct / seen:.4f}   |   val loss {val_loss:.4f}  acc {val_acc:.4f}")
    sync()
    seconds = time.perf_counter() - start  # includes the validation pass after each epoch
    model.load_state_dict(best_state)
    print(f"Training time (s): {seconds:.1f}   best validation epoch: {best_epoch}")
    return pd.DataFrame(history), seconds, best_epoch

In [ ]:
histories, train_seconds, best_epochs = {}, {}, {}
for name, model in MODELS.items():
    print(f"--------\n{name}\n--------")
    histories[name], train_seconds[name], best_epochs[name] = train(model)
    torch.save(model.state_dict(), f"{name.lower()}.pt")
    histories[name].to_csv(f"{RESULT_DIR}/{name.lower()}_history.csv", index=False)

### Training and validation loss and accuracy versus epochs, for each model

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row, (name, h) in enumerate(histories.items()):
    for col, metric in enumerate(("loss", "acc")):
        ax = axes[row, col]
        ax.plot(h.epoch, h[f"train_{metric}"], "--", label="Training")
        ax.plot(h.epoch, h[f"val_{metric}"], label="Validation")
        ax.axvline(best_epochs[name], color="gray", linestyle=":", label="Best validation epoch")
        ax.set_title(f"{name}: {'cross-entropy loss' if metric == 'loss' else 'accuracy'}")
        ax.set_xlabel("Epoch")
axes[0, 0].legend(frameon=False)
plt.tight_layout()
plt.show()

### Comparative plots showing the learning behavior of both architectures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in histories.items():
    for ax, metric in zip(axes, ("loss", "acc")):
        line, = ax.plot(h.epoch, h[f"val_{metric}"], label=f"{name}: validation")
        ax.plot(h.epoch, h[f"train_{metric}"], "--", color=line.get_color(), label=f"{name}: training")
axes[0].set_ylabel("Cross-entropy loss (log scale)")
axes[0].set_yscale("log")
axes[1].set_ylabel("Accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
axes[1].legend(frameon=False)
plt.show()

### Required investigation
> Overfitting rule used below: validation loss ends more than 10% above its own minimum while training accuracy is higher than validation accuracy.

In [ ]:
TARGET = 0.98
for name, h in histories.items():
    reached = h[h.val_acc >= TARGET].epoch
    best_loss_epoch = int(h.loc[h.val_loss.idxmin(), "epoch"])
    gap = h.train_acc.iloc[-1] - h.val_acc.iloc[-1]
    rise = h.val_loss.iloc[-1] - h.val_loss.min()
    print(f"--------\n{name}\n--------")
    print(f"Validation accuracy after epoch 1:         {h.val_acc.iloc[0]:.4f}")
    print(f"First epoch with validation acc >= {TARGET}:  {reached.iloc[0] if len(reached) else 'never'}")
    print(f"Best validation accuracy (epoch):          {h.val_acc.max():.4f} ({best_epochs[name]})")
    print(f"Lowest validation loss (epoch):            {h.val_loss.min():.4f} ({best_loss_epoch})")
    print(f"Final train - validation accuracy gap:     {gap:+.4f}")
    print(f"Validation loss rise after its minimum:    {rise:+.4f}")
    print(f"Std of validation accuracy, last 5 epochs: {h.val_acc.iloc[-5:].std():.4f}")
    if rise > 0.1 * h.val_loss.min() and gap > 0:
        print(f"Verdict: overfitting after epoch {best_loss_epoch}; validation loss rises while training keeps improving.")
    elif h.train_acc.iloc[-1] < TARGET:
        print("Verdict: underfitting; training accuracy itself stays low.")
    else:
        print("Verdict: no clear overfitting or underfitting within the trained epochs.")

first = {n: h.val_acc.iloc[0] for n, h in histories.items()}
spread = {n: h.val_acc.iloc[-5:].std() for n, h in histories.items()}
print(f"\nFaster learner (higher validation accuracy after one epoch):     {max(first, key=first.get)}")
print(f"More stable learner (smaller validation spread, last 5 epochs):  {min(spread, key=spread.get)}")

> How architectural and training decisions may influence convergence (possible explanation, not tested here): the CNN starts with a useful bias (local, shared filters), so each filter learns from every position of every image; the MLP must learn a separate weight for every pixel and has enough parameters (235,146) to memorize details of the 55,000 training images.

> A fixed learning rate with no dropout or weight decay leaves both models free to overfit, which is why the best validation epoch, not the last epoch, is kept for evaluation.

## 4. Quantitative Performance Evaluation

### 1 to 4. Accuracy, precision, recall and F1-score on the independent test set

In [ ]:
probs = {name: predict(model, X_test) for name, model in MODELS.items()}
preds = {name: p.argmax(1).numpy() for name, p in probs.items()}
reports = {name: pd.DataFrame(classification_report(y_true, preds[name], digits=4, output_dict=True, zero_division=0)).T for name in MODELS}

summary = pd.DataFrame({name: {
    "Accuracy": accuracy_score(y_true, preds[name]),
    "Macro precision": r.loc["macro avg", "precision"],
    "Macro recall": r.loc["macro avg", "recall"],
    "Macro F1": r.loc["macro avg", "f1-score"],
    "Weighted F1": r.loc["weighted avg", "f1-score"],
} for name, r in reports.items()}).T
summary.to_csv(f"{RESULT_DIR}/test_summary.csv")
summary.round(4)

### Precision, recall and F1-score for each digit class (0 to 9)

In [ ]:
DIGITS = [str(d) for d in range(10)]
per_class = pd.concat({name: r.loc[DIGITS, ["precision", "recall", "f1-score"]] for name, r in reports.items()}, axis=1)
per_class["F1 difference (CNN - MLP)"] = per_class[("CNN", "f1-score")] - per_class[("MLP", "f1-score")]
per_class["Test images"] = reports["MLP"].loc[DIGITS, "support"].astype(int)
per_class.to_csv(f"{RESULT_DIR}/per_class.csv")
per_class.round(4)

### 5. Confusion matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, name in zip(axes, MODELS):
    ConfusionMatrixDisplay(confusion_matrix(y_true, preds[name]), display_labels=range(10)).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}: test confusion matrix (rows true, columns predicted)")
plt.tight_layout()
plt.show()

### Required investigation

In [ ]:
f1 = per_class.xs("f1-score", axis=1, level=1)
better = summary["Macro F1"].idxmax()
print(f"Better overall model (macro F1): {better}")
print("Test errors:", ", ".join(f"{n} {(preds[n] != y_true).sum()}" for n in MODELS))
for name in MODELS:
    r = reports[name].loc[DIGITS]
    cm = confusion_matrix(y_true, preds[name])
    np.fill_diagonal(cm, 0)
    pairs = [divmod(int(k), 10) for k in np.argsort(cm.ravel())[::-1][:5]]
    print(f"--------\n{name}\n--------")
    print(f"Hardest digits (lowest F1):    {', '.join(f1[name].nsmallest(3).index)}   (F1 from {f1[name].min():.4f} to {f1[name].max():.4f})")
    print(f"Lowest precision (over-predicted digit): {r.precision.idxmin()}   lowest recall (most missed digit): {r.recall.idxmin()}")
    print("Most common errors (true -> predicted):  " + ", ".join(f"{t}->{p} ({cm[t, p]})" for t, p in pairs))

> What precision, recall and F1 reveal that accuracy cannot: accuracy is one number for all digits; precision shows which digit absorbs wrong predictions, recall shows which digit gets missed, and F1 combines both per digit. MNIST classes are nearly balanced, so macro and weighted averages are close here; on imbalanced data they would diverge.

> Connection to the architectures (possible explanation): confusions between digits that share strokes and differ in a small local detail (for example a closed or open top on 4 and 9) favour the CNN's local filters over the MLP's whole-image templates; the confusion pairs printed above show whether this happens here.

## 5. Internal Network Visualization: Layer-by-Layer Analysis

> One function draws every layer, choosing the plot from the output shape: one-channel images in gray, C × H × W feature maps as the first 16 channels (storage order, no cherry-picking), long vectors as a bar chart plus the same values on a square grid, and the 10 logits or probabilities as bars. Every title gives the model, layer, output shape and test image.

In [ ]:
SAMPLE = 0  # test image #0, used for both models


def show_layers(model, name, idx):
    with torch.no_grad():
        outs = layer_outputs(model.eval(), X_test[idx:idx + 1].to(device))
    for layer, out in outs:
        out = out[0].cpu()
        tag = f"{name} | {layer} | output {tuple(out.shape)} | test #{idx}, label {y_test[idx].item()}"
        if out.dim() == 3 and len(out) == 1:  # one-channel image
            fig, ax = plt.subplots(figsize=(2.2, 2.2))
            ax.imshow(out[0], cmap="gray")
            ax.axis("off")
        elif out.dim() == 3:  # feature maps
            n = min(16, len(out))
            tag += f" | zeros {(out == 0).float().mean().item():.0%} | channels 0 to {n - 1} shown"
            fig, axes = plt.subplots(1, n, figsize=(14, 1.4))
            for c, ax in enumerate(axes):
                ax.imshow(out[c], cmap="viridis")
                ax.set_title(f"ch {c}", fontsize=7)
                ax.axis("off")
        elif len(out) == 10:  # logits or probabilities
            tag += f" | predicted {out.argmax().item()}"
            fig, ax = plt.subplots(figsize=(5, 2))
            ax.bar(range(10), out.numpy())
            ax.set_xticks(range(10))
            ax.set_xlabel("Digit")
        else:  # vector: bars and the same values on a square grid
            tag += f" | zeros {(out == 0).float().mean().item():.0%}"
            side = int(np.ceil(np.sqrt(len(out))))
            grid = torch.full((side * side,), float("nan"))
            grid[:len(out)] = out
            fig, axes = plt.subplots(1, 2, figsize=(12, 2), gridspec_kw={"width_ratios": [5, 1]})
            axes[0].bar(range(len(out)), out.numpy(), width=1.0)
            axes[0].set_xlim(0, len(out))
            axes[0].set_xlabel("Unit")
            axes[1].imshow(grid.view(side, side), cmap="viridis")
            axes[1].axis("off")
        fig.suptitle(tag, fontsize=9, y=1.15 if out.dim() == 3 and len(out) > 1 else 1.02)
        plt.show()


def collect(model, X):
    with torch.no_grad():
        return {layer: out.flatten(1).cpu() for layer, out in layer_outputs(model.eval(), X.to(device))}


SUBSET = 1000  # first 1,000 test images for activation statistics
labels = y_test[:SUBSET]
acts = {name: collect(model, X_test[:SUBSET]) for name, model in MODELS.items()}

### A. MLP internal visualization

#### Input, flattening, every hidden layer, output scores and probabilities

In [ ]:
show_layers(mlp, "MLP", SAMPLE)

#### Organization and distribution of the hidden activations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6.5))
for col, layer in enumerate(("relu1", "relu2")):
    a = acts["MLP"][layer]
    axes[0, col].hist(a[a > 0].numpy(), bins=60)
    axes[0, col].set_title(f"MLP | {layer} | {a.shape[1]} units | {SUBSET} test images\n"
                           f"zeros {(a == 0).float().mean().item():.0%}, units never active {(a.max(0).values == 0).float().mean().item():.0%}", fontsize=9)
    axes[0, col].set_xlabel("Activation (non-zero values only)")
    means = torch.stack([a[labels == d].mean(0) for d in range(10)])  # (10 digits, units)
    axes[1, col].imshow(means[:, means.argmax(0).argsort()], aspect="auto", cmap="viridis")
    axes[1, col].set_yticks(range(10))
    axes[1, col].set_ylabel("Digit")
    axes[1, col].set_xlabel("Unit, sorted by the digit it responds to most")
    axes[1, col].set_title(f"MLP | {layer} | mean activation per digit", fontsize=9)
plt.tight_layout()
plt.show()

#### Learned weights and their relationship to the input pixels

In [ ]:
W1 = mlp.fc1.weight.detach().cpu()  # (256, 784)
fig, axes = plt.subplots(4, 8, figsize=(10, 5.4))
for i, ax in enumerate(axes.ravel()):
    lim = W1[i].abs().max()
    ax.imshow(W1[i].view(28, 28), cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(f"unit {i}", fontsize=7)
    ax.axis("off")
fig.suptitle("MLP | fc1 weights of units 0 to 31 (of 256), each reshaped to 28 × 28 | red positive, blue negative", fontsize=10)
plt.show()

W3 = mlp.fc3.weight.detach().cpu()  # (10, 128)
plt.figure(figsize=(12, 2.6))
plt.imshow(W3, aspect="auto", cmap="RdBu_r", vmin=-W3.abs().max(), vmax=W3.abs().max())
plt.yticks(range(10))
plt.ylabel("Digit")
plt.xlabel("relu2 unit")
plt.colorbar()
plt.title("MLP | fc3 weights (10 × 128): how each relu2 unit votes for each digit", fontsize=10)
plt.show()

#### Interpretation: how information is represented in the MLP
`input`, `norm`, `flatten`: reveal that the image becomes 784 numbers in row order; the square grid of the flattened vector is the image again, so flattening loses no values, only the 2D neighbourhood. Cannot reveal anything learned (no parameters).

`fc1`, `relu1`: each unit is a weighted vote of all 784 pixels; ReLU silences units with negative evidence (zeros % in the title), so a digit is coded by a sparse set of active units. Cannot reveal what one unit "means": the code is distributed over many units.

`fc1` weights: whole-image templates of positive and negative pixel regions, by construction never local. Cannot reveal how templates combine; a readable template is not proof of what the unit detects.

`fc2`, `relu2`, per-digit heat maps: show how activity is organized by digit at each depth (the nearest-centroid table in C measures this). Cannot reveal cause and effect; they are averages over 1,000 images.

`fc3`, `softmax`: show the decision and its confidence. A high probability does not mean a correct or calibrated prediction (Section 6).

### B. CNN internal visualization

#### Learned convolutional kernels

In [ ]:
K1 = cnn.conv1.weight.detach().cpu()  # (16, 1, 3, 3)
fig, axes = plt.subplots(1, 16, figsize=(14, 1.4))
for c, ax in enumerate(axes):
    lim = K1[c].abs().max()
    ax.imshow(K1[c, 0], cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_title(f"k{c}", fontsize=7)
    ax.axis("off")
fig.suptitle("CNN | conv1 kernels (16 × 1 × 3 × 3) | red positive, blue negative", fontsize=9, y=1.15)
plt.show()

K2 = cnn.conv2.weight.detach().cpu()  # (32, 16, 3, 3)
fig, axes = plt.subplots(8, 16, figsize=(12, 6.5))
for o in range(8):
    lim = K2[o].abs().max()
    for i in range(16):
        ax = axes[o, i]
        ax.imshow(K2[o, i], cmap="RdBu_r", vmin=-lim, vmax=lim)
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_ylabel(f"out {o}", fontsize=7)
        if o == 0:
            ax.set_title(f"in {i}", fontsize=7)
fig.suptitle("CNN | conv2 kernels: output filters 0 to 7 (of 32) × all 16 input channels", fontsize=10)
plt.show()

#### Feature maps, activations, pooling, later layers and final outputs

In [ ]:
show_layers(cnn, "CNN", SAMPLE)

#### The effect of ReLU and max-pooling on the same channels

In [ ]:
with torch.no_grad():
    outs = dict(layer_outputs(cnn.eval(), X_test[SAMPLE:SAMPLE + 1].to(device)))
fig, axes = plt.subplots(3, 8, figsize=(12, 4.8))
for row, layer in enumerate(("conv1", "relu1", "pool1")):
    out = outs[layer][0].cpu()
    for c in range(8):
        ax = axes[row, c]
        ax.imshow(out[c], cmap="viridis")
        ax.set_xticks([])
        ax.set_yticks([])
        if c == 0:
            ax.set_ylabel(f"{layer}\n{tuple(out.shape)}", fontsize=8)
        if row == 0:
            ax.set_title(f"ch {c}", fontsize=8)
fig.suptitle(f"CNN | conv1, after ReLU, after max-pooling, channels 0 to 7 | test #{SAMPLE}, label {y_test[SAMPLE].item()}", fontsize=10)
plt.show()
print(f"Values set to zero by relu1: {(outs['relu1'] == 0).float().mean().item():.0%}   relu2: {(outs['relu2'] == 0).float().mean().item():.0%}   relu3: {(outs['relu3'] == 0).float().mean().item():.0%}")

#### Changes in spatial dimensions and channel depth

In [ ]:
shapes = layer_table(cnn)[["Output shape"]].copy()
shapes["Values per image"] = [int(np.prod(s)) for s in shapes["Output shape"]]
shapes.T

#### Interpretation: how the CNN transforms the image
`conv1` kernels: 3 × 3 contrast patterns; a kernel with a positive and a negative side responds to edges at that orientation. Cannot reveal on its own what the network builds from them; 3 × 3 is too small to read meaning.

`conv1`, `relu1`: 16 maps of the same 28 × 28 digit, each highlighting different stroke edges or regions, with the spatial layout kept. ReLU keeps only where a pattern is present (zeros % printed). Cannot reveal a channel's role from one image.

`pool1`, `pool2`: halve the resolution and keep the strongest response in each 2 × 2 window, so a 1-pixel shift of a stroke often gives the same output (local shift tolerance).

`conv2`, `relu2`, `pool2`: 32 coarser 7 × 7 maps that respond to combinations of edges (corners, stroke ends, curves) over a 10 × 10 receptive field. Cannot reveal exactly which combination without testing many images per channel.

`flatten`, `fc1`, `relu3`, `fc2`, `softmax`: the 1568 extracted features become a 64-value code and then the decision; the spatial layout is only dropped after features have been extracted. The same caveat on confidence as for the MLP applies.

### C. Comparative interpretation (same input images for both models)

In [ ]:
COMPARE = [0, 1, 2]  # the same test images for both models


def progression(model, name, idxs):
    with torch.no_grad():
        runs = [layer_outputs(model.eval(), X_test[i:i + 1].to(device)) for i in idxs]
    n = len(runs[0])
    fig, axes = plt.subplots(len(idxs), n, figsize=(1.35 * n, 1.7 * len(idxs) + 0.6), squeeze=False)
    for r, (i, outs) in enumerate(zip(idxs, runs)):
        for c, (layer, out) in enumerate(outs):
            ax, out = axes[r, c], out[0].cpu()
            if out.dim() == 3:
                ax.imshow(out.mean(0), cmap="gray" if c == 0 else "viridis")  # channel mean
            elif len(out) == 10:
                ax.bar(range(10), out.numpy())
                ax.set_xlabel(f"pred {out.argmax().item()}", fontsize=7)
            else:
                side = int(np.ceil(np.sqrt(len(out))))
                grid = torch.full((side * side,), float("nan"))
                grid[:len(out)] = out
                ax.imshow(grid.view(side, side), cmap="viridis")
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(f"{layer}\n{'×'.join(map(str, out.shape))}", fontsize=7)
            if c == 0:
                ax.set_ylabel(f"#{i}, label {y_test[i].item()}", fontsize=7)
    fig.suptitle(f"{name}: each row is one test image through every layer (feature maps shown as their channel mean)", fontsize=10)
    plt.tight_layout()
    plt.show()


for name, model in MODELS.items():
    progression(model, name, COMPARE)

#### How class separation changes from layer to layer (nearest-centroid accuracy: centroids from images 0 to 499, tested on images 500 to 999)

In [ ]:
def centroid_accuracy(feats, y):
    half = len(y) // 2
    centroids = torch.stack([feats[:half][y[:half] == d].mean(0) for d in range(10)])
    return (torch.cdist(feats[half:], centroids).argmin(1) == y[half:]).float().mean().item()


for name in MODELS:
    sep = pd.Series({layer: centroid_accuracy(f, labels) for layer, f in acts[name].items()})
    display(sep.to_frame(f"{name}: nearest-centroid accuracy").T.round(3))
    print(f"{name}: separability goes from {sep.iloc[0]:.3f} (input) to {sep.iloc[-1]:.3f} (softmax); largest gain at layer {sep.diff().idxmax()}")

In [ ]:
PCA_LAYERS = {"MLP": ["input", "relu1", "relu2", "fc3"], "CNN": ["input", "relu1", "pool2", "fc2"]}
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for row, name in enumerate(MODELS):
    for ax, layer in zip(axes[row], PCA_LAYERS[name]):
        z = PCA(2).fit_transform(acts[name][layer].numpy())
        points = ax.scatter(z[:, 0], z[:, 1], c=labels.numpy(), cmap="tab10", s=4)
        ax.set_title(f"{name} | {layer} | {acts[name][layer].shape[1]} values", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
fig.colorbar(points, ax=axes, ticks=range(10), label="Digit")
fig.suptitle(f"First two principal components of each layer's output, first {SUBSET} test images")
plt.show()

> How the input evolves: in the MLP the image is a vector from the first layer on, and each dense layer re-mixes all values; in the CNN it stays an image (now with 16, then 32 channels) through both convolution blocks and only becomes a vector after features are extracted.

> Dense activations vs feature maps: an MLP activation says "how much of a whole-image template is present"; a CNN feature map says "where a local pattern is present", so CNN maps can be read as images and MLP activations cannot.

> Earlier vs deeper layers: early CNN maps follow the digit's strokes closely; deeper maps are coarser and more abstract. In both networks the deeper layers group images by digit more clearly, which the nearest-centroid accuracy and the PCA plots measure rather than assume.

> Do the visualizations explain predictions and errors? They show where the evidence for a class came from and when it was weak (flat probabilities, few active units), which Section 6 uses on wrong predictions. They cannot prove why a unit fires: they are correlations on a few images, and channel means or 16-channel subsets hide part of the representation.

## 6. Misclassification and Prediction Analysis

In [ ]:
wrong = {n: preds[n] != y_true for n in MODELS}
conf = {n: probs[n].max(1).values.numpy() for n in MODELS}
both_right = np.where(~wrong["MLP"] & ~wrong["CNN"])[0]
both_wrong = np.where(wrong["MLP"] & wrong["CNN"])[0]
mlp_only = np.where(wrong["MLP"] & ~wrong["CNN"])[0]
cnn_only = np.where(~wrong["MLP"] & wrong["CNN"])[0]


def gallery(idxs, title):
    idxs = list(idxs)[:8]
    if not idxs:
        print(f"{title}: none")
        return
    fig, axes = plt.subplots(1, len(idxs), figsize=(1.9 * len(idxs), 2.9), squeeze=False)
    for ax, i in zip(axes[0], idxs):
        ax.imshow(X_test[i, 0], cmap="gray")
        ax.set_title(f"#{i}  true {y_true[i]}\nMLP {preds['MLP'][i]} ({conf['MLP'][i]:.2f})\nCNN {preds['CNN'][i]} ({conf['CNN'][i]:.2f})", fontsize=8)
        ax.axis("off")
    fig.suptitle(title, y=1.08)
    plt.show()

### Examples of correctly classified images (true label, predicted label and confidence)

In [ ]:
gallery(both_right, "Correct for both models")

### Examples of incorrectly classified images

In [ ]:
gallery(both_wrong, "Wrong for both models")
gallery(mlp_only, "Wrong for the MLP only")
gallery(cnn_only, "Wrong for the CNN only")

### Prediction confidence and class-probability distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, n in zip(axes, MODELS):
    ax.hist(conf[n][~wrong[n]], bins=40, range=(0, 1), alpha=0.7, label="Correct")
    ax.hist(conf[n][wrong[n]], bins=40, range=(0, 1), alpha=0.7, label="Wrong")
    ax.set_yscale("log")
    ax.set_title(f"{n}: confidence (highest softmax probability), 10,000 test images", fontsize=10)
    ax.set_xlabel("Confidence")
axes[0].legend(frameon=False)
plt.show()

ERRORS = list(both_wrong[:2]) + list(mlp_only[:1]) + list(cnn_only[:1])
for i in ERRORS:
    fig, axes = plt.subplots(1, 3, figsize=(11, 2), gridspec_kw={"width_ratios": [1, 3, 3]})
    axes[0].imshow(X_test[i, 0], cmap="gray")
    axes[0].set_title(f"#{i}, true {y_true[i]}", fontsize=9)
    axes[0].axis("off")
    for ax, n in zip(axes[1:], MODELS):
        colors = ["tab:green" if d == y_true[i] else "tab:red" if d == preds[n][i] else "tab:gray" for d in range(10)]
        ax.bar(range(10), probs[n][i].numpy(), color=colors)
        ax.set_xticks(range(10))
        ax.set_ylim(0, 1)
        ax.set_title(f"{n}: predicted {preds[n][i]} ({conf[n][i]:.2f})  green true, red predicted", fontsize=9)
    plt.show()

### A comparison of errors made by the two architectures

In [ ]:
agreement = pd.DataFrame([[len(both_right), len(cnn_only)], [len(mlp_only), len(both_wrong)]],
                         index=["MLP correct", "MLP wrong"], columns=["CNN correct", "CNN wrong"])
display(agreement)

b, c = len(mlp_only), len(cnn_only)
p_value = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0  # exact McNemar test on the images where the models disagree
print(f"McNemar P-Value:   {p_value:.4g}")
if p_value > 0.05:
    print("At the 95% confidence level, there is not enough evidence to reject the null hypothesis; the difference in error rates may be chance.")
else:
    print(f"At the 95% confidence level, there is enough evidence to reject the null hypothesis; the {'CNN' if b > c else 'MLP'} makes significantly fewer errors.")

### Questions to investigate

In [ ]:
print(f"1. Images both models classify incorrectly:   {len(both_wrong)}")
print(f"2. Wrong for the MLP only / the CNN only:     {len(mlp_only)} / {len(cnn_only)}")
for n in MODELS:
    print(f"3. {n} errors with confidence above 0.9:      {(wrong[n] & (conf[n] > 0.9)).sum()} of {wrong[n].sum()}"
          f"   (mean confidence: correct {conf[n][~wrong[n]].mean():.3f}, wrong {conf[n][wrong[n]].mean():.3f})")

#### 4. Can the internal layer visualizations help explain particular prediction errors?

In [ ]:
ERR = ERRORS[0] if ERRORS else SAMPLE
for name, model in MODELS.items():
    progression(model, name, [ERR])

> The layer views of a wrong image show where the evidence went astray (for example CNN maps firing on a stroke that resembles another digit, or the MLP spreading its probability over two digits), which makes an error easier to understand; they cannot prove the cause, because they are correlations on a single image.

#### 5. What do these examples reveal about the strengths and limitations of each model?
> Images both models get wrong point to ambiguity in the data itself (check the gallery: some are hard even for a human); images only one model gets wrong expose that architecture's weakness. Errors made with confidence above 0.9 show that neither model knows when it is wrong, so a softmax probability is not a safe reject signal on its own.

## 7. Model Complexity and Computational Efficiency

### Hardware and measurement procedure

In [ ]:
print("Platform:   ", platform.platform())
print("Processor:  ", platform.processor() or platform.machine())
print("CPU cores:  ", os.cpu_count())
print("Device:     ", torch.cuda.get_device_name(0) if device.type == "cuda" else device.type)
print("PyTorch:    ", torch.__version__)

> Training time: wall clock for all epochs, including the validation pass after each epoch, with the device synchronized before and after. Inference time: mean of 20 timed forward passes after 5 warm-up passes, device synchronized, data already on the device, at batch size 1 (latency) and 1,000 (throughput). One measurement each; rerun to see the spread.

### Parameters, model size, training time, inference time and computation

In [ ]:
def macs(model):
    total, x = 0, X_test[:1].to(device)
    with torch.no_grad():
        for layer in model.children():
            y = layer(x)
            if isinstance(layer, nn.Linear):
                total += layer.in_features * layer.out_features
            elif isinstance(layer, nn.Conv2d):
                total += y[0].numel() * layer.in_channels * layer.kernel_size[0] * layer.kernel_size[1]
            x = y
    return total


@torch.no_grad()
def inference_ms(model, batch_size, repeats=20):
    model.eval()
    xb = X_test[:batch_size].to(device)
    for _ in range(5):
        model(xb)  # warm-up
    sync()
    start = time.perf_counter()
    for _ in range(repeats):
        model(xb)
    sync()
    return 1000 * (time.perf_counter() - start) / repeats / batch_size


complexity = pd.DataFrame({name: {
    "Trainable parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "Model file size (KB)": os.path.getsize(f"{name.lower()}.pt") / 1024,
    "Multiply-accumulates per image": macs(model),
    "Training time (s)": train_seconds[name],
    "Inference, batch 1 (ms per image)": inference_ms(model, 1),
    "Inference, batch 1000 (ms per image)": inference_ms(model, 1000),
    "Test accuracy": summary.loc[name, "Accuracy"],
} for name, model in MODELS.items()}).T
complexity.to_csv(f"{RESULT_DIR}/complexity.csv")
complexity.round(4)

### Does a more complex network necessarily provide better results?

In [ ]:
more_params = complexity["Trainable parameters"].idxmax()
more_compute = complexity["Multiply-accumulates per image"].idxmax()
best = complexity["Test accuracy"].idxmax()
print(f"More parameters: {more_params}   more computation: {more_compute}   higher test accuracy: {best}")
if more_params == best:
    print("The model with more parameters is also the more accurate one.")
else:
    print(f"More parameters did not give better results: the {best} is more accurate with fewer parameters.")

> Trade-offs: by construction the CNN has 2.2 times fewer parameters (105,866 against 235,146) and a smaller file, but needs 4.8 times more multiply-accumulates per image (1,117,056 against 234,752), because each kernel weight is reused at all positions. "Complexity" therefore has two meanings: memory (parameters) favours the CNN, arithmetic favours the MLP; the measured times show how much of the extra arithmetic the hardware hides through parallelism.

## 8. Generalization and Robustness

#### Experiment: Gaussian noise, rotation and translation at increasing strength, applied to all 10,000 test images in [0, 1] pixel space (the same noise for both models)

In [ ]:
def corrupt(X, kind, level):
    if kind.startswith("Gaussian"):
        noise = torch.randn(X.shape, generator=torch.Generator().manual_seed(SEED))
        return (X + level * noise).clamp(0, 1)
    if kind.startswith("Rotation"):
        return TF.rotate(X, float(level), interpolation=TF.InterpolationMode.BILINEAR)
    return TF.affine(X, angle=0.0, translate=[int(level), 0], scale=1.0, shear=[0.0])  # shift right, empty pixels black


LEVELS = {"Gaussian noise (std)": [0, 0.1, 0.2, 0.3, 0.4, 0.5],
          "Rotation (degrees)": [0, 10, 20, 30, 45, 60],
          "Translation (pixels to the right)": [0, 1, 2, 3, 4, 6]}

fig, axes = plt.subplots(3, 6, figsize=(10, 5.6))
for row, (kind, levels) in enumerate(LEVELS.items()):
    for col, level in enumerate(levels):
        axes[row, col].imshow(corrupt(X_test[SAMPLE:SAMPLE + 1], kind, level)[0, 0], cmap="gray")
        axes[row, col].set_title(f"{kind.split(' (')[0]}: {level}", fontsize=8)
        axes[row, col].axis("off")
fig.suptitle(f"Test #{SAMPLE}, label {y_test[SAMPLE].item()}, at each level")
plt.show()

In [ ]:
rows = []
for kind, levels in LEVELS.items():
    for level in levels:
        X_mod = corrupt(X_test, kind, level)
        for name, model in MODELS.items():
            rows.append({"Transform": kind, "Level": level, "Model": name,
                         "Accuracy": (predict(model, X_mod).argmax(1) == y_test).float().mean().item()})
robust = pd.DataFrame(rows)
robust.to_csv(f"{RESULT_DIR}/robustness.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, kind in zip(axes, LEVELS):
    for name in MODELS:
        r = robust[(robust.Transform == kind) & (robust.Model == name)]
        ax.plot(r.Level, r.Accuracy, marker="o", label=name)
    ax.set_xlabel(kind)
    ax.set_ylabel("Test accuracy")
    ax.set_ylim(0, 1)
axes[0].legend(frameon=False)
plt.show()

robust.pivot_table(index=["Transform", "Level"], columns="Model", values="Accuracy").round(4)

### Do the two architectures respond differently to the modifications?

In [ ]:
for kind in LEVELS:
    r = robust[robust.Transform == kind].pivot(index="Level", columns="Model", values="Accuracy")
    drop = r.iloc[0] - r.iloc[-1]
    print(f"--------\n{kind}\n--------")
    print(f"Original images:     MLP {r.iloc[0]['MLP']:.4f}   CNN {r.iloc[0]['CNN']:.4f}")
    print(f"Strongest level {r.index[-1]:g}: MLP {r.iloc[-1]['MLP']:.4f}   CNN {r.iloc[-1]['CNN']:.4f}")
    print(f"Accuracy drop:       MLP {drop['MLP']:.4f}   CNN {drop['CNN']:.4f}")
    print(f"More robust (smaller drop): {drop.idxmin()}")

> Relation to the architectures: translation tests the built-in assumptions most directly. Convolution is shift equivariant and each max-pool absorbs about one pixel of shift, but the dense `fc1` after `pool2` is tied to positions, so the CNN is only partly shift tolerant; the MLP has a separate weight for every pixel, so a shifted digit lands on weights that never saw it. Neither architecture has rotation built in, so any rotation tolerance comes from the natural slant variation in the training digits. Pixel noise is not spatial; 3 × 3 kernels average neighbouring pixels, which may smooth it (possible explanation, to be checked against the drops above).

> Evidence of robustness: the printed drops show which model degrades less for each transformation on this test set. They are evidence of robustness to these three synthetic transformations only, from one seed; they do not show robustness to other shifts (new handwriting styles, adversarial noise).

## 9. Experimental Fairness and Reproducibility

### Dataset preparation and partitioning; no leakage between partitions

In [ ]:
train_hashes = {train_full.data[i].numpy().tobytes() for i in train_idx.tolist()}
val_hashes = {train_full.data[i].numpy().tobytes() for i in val_idx.tolist()}
test_hashes = {img.numpy().tobytes() for img in test_set.data}
print(f"Shared indices, training and validation:     {len(set(train_idx.tolist()) & set(val_idx.tolist()))}")
print(f"Identical images, training and validation:   {len(train_hashes & val_hashes)}")
print(f"Identical images, training and test:         {len(train_hashes & test_hashes)}")

distribution = pd.DataFrame({name: pd.Series(y.numpy()).value_counts(normalize=True).sort_index()
                             for name, y in (("Training", y_train), ("Validation", y_val), ("Testing", y_test))})
distribution.T.round(3)

### Training configurations, hyperparameters and model selection

In [ ]:
config = pd.Series({
    "Data": f"MNIST, {len(y_train):,} training / {len(y_val):,} validation (seeded random split of the 60,000) / {len(y_test):,} official test",
    "Preprocessing": f"Pixels scaled to [0, 1], standardized inside the model with training-split mean {MEAN:.4f} and std {STD:.4f}",
    "Augmentation": "None",
    "Optimizer": f"Adam, learning rate {LR}, no weight decay",
    "Batch size / epochs": f"{BATCH_SIZE} / {EPOCHS}",
    "Loss": "Cross-entropy",
    "Regularization": "None (no dropout, weight decay or batch normalization)",
    "Batch order": "Identical for both models (same seeded generator)",
    "Initialization": f"PyTorch default, seed {SEED} set before building each model",
    "Model selection": "Epoch with the highest validation accuracy; the test set is used once, after selection",
}, name="Both models")
config.to_frame()

### Computational environment and implementation details

In [ ]:
print("Python:       ", platform.python_version())
print("PyTorch:      ", torch.__version__, "  torchvision:", torchvision.__version__)
print("NumPy:        ", np.__version__, "  scikit-learn:", sklearn.__version__, "  pandas:", pd.__version__)
print("Device:       ", torch.cuda.get_device_name(0) if device.type == "cuda" else device.type)
print("cuDNN deterministic:", torch.backends.cudnn.deterministic)

### Factors that may affect the reliability or reproducibility of the results
> Differences in training conditions: none; both models share data, batch order, optimizer, learning rate, epochs and selection rule. The architectures differ in size on purpose (the CNN has fewer parameters but more computation), so a win cannot be credited to model size alone.

> One seed: run-to-run variation from initialization is not measured, so a difference of a few test errors may be noise (the McNemar test in Section 6 checks this for the error counts).

> GPU kernels can be non-deterministic even with fixed seeds, and timings depend on the hardware and background load.

> No hyperparameter tuning per model: a tuned or regularized MLP (dropout, augmentation) would likely narrow the gap, so the comparison is between these two configurations, not the best possible MLP and CNN.

## 10. Critical Discussion and Final Conclusions

#### Evidence table used for the answers below

In [ ]:
final = pd.DataFrame({name: {
    "Test accuracy": summary.loc[name, "Accuracy"],
    "Macro F1": summary.loc[name, "Macro F1"],
    "Best validation epoch": best_epochs[name],
    "Final train - validation accuracy gap": histories[name].train_acc.iloc[-1] - histories[name].val_acc.iloc[-1],
    "Test errors": int(wrong[name].sum()),
    "Errors with confidence above 0.9": int((wrong[name] & (conf[name] > 0.9)).sum()),
    "Trainable parameters": complexity.loc[name, "Trainable parameters"],
    "Multiply-accumulates per image": complexity.loc[name, "Multiply-accumulates per image"],
    "Training time (s)": train_seconds[name],
    **{f"Accuracy at strongest {kind.split(' (')[0].lower()}": robust[(robust.Transform == kind) & (robust.Model == name)].Accuracy.iloc[-1] for kind in LEVELS},
} for name in MODELS})
final.to_csv(f"{RESULT_DIR}/final_comparison.csv")
final.round(4)

### 1. Architectural understanding: what are the fundamental differences between how MLPs and CNNs learn from image data?
> The MLP treats the image as an unordered list of 784 numbers and learns one weight per pixel per unit, so a pattern learned in one place must be learned again in every other place. The CNN learns small shared filters that slide over the image, so a stroke detector learned anywhere works everywhere, and neighbourhoods are kept until features are extracted.

### 2. Learning behavior: what do the training and validation curves reveal about the optimization and generalization behavior of each architecture?
> Training loss shows optimization and validation loss shows generalization. The Section 3 verdicts give which model learns faster, which is more stable and where validation loss turns upward while training loss keeps falling (overfitting); with no regularization the larger train-validation gap marks the model that memorizes more, and keeping the best validation epoch is what protects the evaluated models.

### 3. Internal representations: what did you observe inside the layers of both networks? How does their feature representation differ?
> MLP hidden layers are vectors with no spatial meaning: whole-image templates in `fc1`, then sparser, more digit-specific codes. CNN layers keep spatial maps: edge-like responses in `conv1`, coarser combinations in `conv2`, and only the last dense layers become digit-specific. The nearest-centroid accuracy and PCA plots in Section 5C measure how quickly each network separates the digits with depth.

### 4. Performance: which model performed better, and what experimental evidence explains the difference?
> The Section 4 summary and the McNemar test in Section 6 give the better model and whether the gap is larger than chance; the per-class F1 table and the most common confusion pairs show which digits create the gap, and the robustness drops in Section 8 show whether the gap grows when local shape matters more than exact pixel position.

### 5. Computational efficiency: how do the architectures compare in terms of parameters, model size and execution time?
> The CNN has 2.2 times fewer parameters and a smaller file but 4.8 times more multiply-accumulates per image; Section 7 reports how this turns into measured training and inference time on this hardware, where parallel convolution kernels can hide much of the extra arithmetic.

### 6. Error analysis: what can be learned from the incorrect predictions made by each model?
> Shared errors point to ambiguous or badly written digits; errors made by one model only point to that architecture's weakness, and the confusion pairs show which shape differences each model misses. High-confidence errors show that softmax confidence is not a calibrated measure of correctness.

### 7. Generalization: how does each model respond to changes in the input images, and what does this reveal about its learned representations?
> The accuracy drops under shift and rotation reveal whether a model learned position-specific pixel templates (sharp drop with small shifts) or local shape features that survive small moves; the noise results show how much each model relies on exact pixel values. Section 8 prints which model degrades less for each transformation.

### 8. Engineering trade-offs: under what circumstances might an MLP be preferable to a CNN, or vice versa?
> MLP: tabular or already-extracted feature vectors with no spatial structure, tiny fixed-position inputs, or hardware where arithmetic per prediction is the bottleneck and a small accuracy loss is acceptable. CNN: images and other grid data where patterns move, limited training data (weight sharing reuses what is learned), and tight memory for parameters.

### 9. Experimental limitations: what limitations exist in your study? Which conclusions can be supported by your experiments, and which would require further investigation?
> Supported: a comparison of these two configurations under one shared, leakage-checked training setup on MNIST. Needs further work: claims about MLPs and CNNs in general (one size per family, one seed, no tuning, no augmentation), robustness beyond three synthetic transformations, and timing claims beyond this hardware.

### 10. Lessons learned: what new insights have you gained about neural network architecture design, training, evaluation and interpretation?
> An architecture is a set of assumptions about the data; the CNN's locality and sharing assumptions match images, and that is visible in its layers, not only in its accuracy. Validation curves, per-class metrics, error overlap and layer visualizations each showed something test accuracy hides, and visualizations show correlations, not proof of what a unit computes.

## 11. Final Comparative Summary

### Why do MLPs and CNNs behave differently on the same image classification task, and what can we learn from examining their internal representations?
> They bring different assumptions to the same pixels: the MLP flattens the image and learns position-specific whole-image templates, while the CNN keeps the 2D layout and learns small shared filters that build edges into parts and parts into digits. The internal representations make this visible: CNN feature maps keep spatial structure and become more abstract with depth, while MLP activations are distributed codes with no spatial meaning; the layer measurements, error overlap and robustness tests in this notebook show how these differences turn into the accuracy, efficiency and robustness differences reported above.